# Training
Model dilatih pada GPU Kaggle. Tiga eksperimen dibandingkan, kemudian dua model terbaik digabung (ensemble).

| Eksperimen | Model | Ukuran input (tinggi x lebar) | Perubahan |
|---|---|---|---|
| 1 | ResNet-18 | 224 x 224 | baseline |
| 2 | ConvNeXt-Tiny | 384 x 384 | arsitektur lebih modern, resolusi lebih tinggi |
| 3 | ConvNeXt-Tiny | 512 x 384 | mengikuti rasio halaman (± 1,31), resolusi vertikal lebih tinggi, regularisasi stochastic depth |
| ensemble | 2 + 3 | - | rata-rata probabilitas |

Seluruh model menggunakan bobot pre-trained ImageNet yang di-fine-tune pada semua lapisan. Pengaturan tiap eksperimen ada di folder `configs/`.

## 0. Setup
Jika dijalankan di Kaggle, repository di-clone otomatis ke `/kaggle/working`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Violita-archive/document-image-classification.git"
REPO_NAME = "document-image-classification"

if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working") / REPO_NAME
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    try:
        import timm
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Root:", ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.dataset import clean_train_data, load_metadata, make_loader, split_train_val
from src.model import load_checkpoint
from src.predict import predict_test, save_submission, validate_submission
from src.train import fit
from src.transforms import denormalize, get_train_transforms, to_hw
from src.utils import CLASS_NAMES, ensure_dirs, get_device, load_config, plot_history, resolve_data_dir, set_seed

FIG_DIR, METRIC_DIR, SUB_DIR = Path("results/figures"), Path("results/metrics"), Path("results/submissions")
ensure_dirs(FIG_DIR, METRIC_DIR, SUB_DIR, "checkpoint")

EXPERIMENTS = {
    "baseline_resnet18": "configs/baseline.yaml",
    "convnext_tiny_384": "configs/convnext_384.yaml",
    "convnext_tiny_512": "configs/config.yaml",
}
configs = {name: load_config(path) for name, path in EXPERIMENTS.items()}
final_cfg = configs["convnext_tiny_512"]
set_seed(final_cfg["seed"])

device = get_device()
print("PyTorch", torch.__version__, "| device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
display(pd.DataFrame(configs).T)

## 1. Data
Pembersihan mengikuti hasil EDA: file yang tidak dapat dibuka dan grup gambar identik berlabel berbeda dibuang, sedangkan duplikat berlabel sama disisakan satu. Data bersih kemudian dibagi 90% train dan 10% validation secara stratified. Pembagian yang sama dipakai untuk semua eksperimen.

In [ ]:
DATA_DIR = resolve_data_dir(final_cfg["data_dir"])
print("Folder data:", DATA_DIR)

train_full, test_df = load_metadata(DATA_DIR)
clean_df, clean_summary = clean_train_data(train_full, n_workers=8)
display(pd.Series(clean_summary, name="jumlah").to_frame().T)

tr_df, va_df = split_train_val(clean_df, final_cfg["val_size"], final_cfg["seed"])
print(f"train: {len(tr_df):,} | validation: {len(va_df):,} | test: {len(test_df):,}")
display(pd.DataFrame({"train": tr_df["class_name"].value_counts(), "validation": va_df["class_name"].value_counts()}).reindex(CLASS_NAMES).T)

## 2. Contoh hasil augmentasi

In [ ]:
aug_loader = make_loader(tr_df.sample(8, random_state=1), get_train_transforms(final_cfg["img_size"]), 8, False, 0)
x, y = next(iter(aug_loader))
fig, axes = plt.subplots(1, 8, figsize=(20, 4.2))
for ax, img, lab in zip(axes, x, y):
    ax.imshow(denormalize(img))
    ax.set_title(CLASS_NAMES[lab], fontsize=8)
    ax.axis("off")
h, w = to_hw(final_cfg["img_size"])
plt.suptitle(f"Contoh gambar setelah augmentasi ({h}x{w})")
plt.tight_layout()
plt.savefig(FIG_DIR / "train_augmentation_examples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Eksperimen 1: Baseline ResNet-18 (224 x 224)

In [ ]:
outs = {}
outs["baseline_resnet18"] = fit(configs["baseline_resnet18"], tr_df, va_df, exp_name="baseline_resnet18")
o = outs["baseline_resnet18"]
print(f"Akurasi validation terbaik: {o['best_acc']:.4f} (epoch {o['best_epoch']}), waktu {o['minutes']:.1f} menit")
plot_history(o["history"], "Baseline ResNet-18 (224 x 224)", FIG_DIR / "curve_baseline_resnet18.png")

## 4. Eksperimen 2: ConvNeXt-Tiny (384 x 384)

In [ ]:
outs["convnext_tiny_384"] = fit(configs["convnext_tiny_384"], tr_df, va_df, exp_name="convnext_tiny_384")
o = outs["convnext_tiny_384"]
print(f"Akurasi validation terbaik: {o['best_acc']:.4f} (epoch {o['best_epoch']}), waktu {o['minutes']:.1f} menit")
plot_history(o["history"], "ConvNeXt-Tiny (384 x 384)", FIG_DIR / "curve_convnext_tiny_384.png")

## 5. Eksperimen 3: ConvNeXt-Tiny (512 x 384)
Pada eksperimen 2 akurasi train mencapai ±99% sedangkan validation ±89%, sehingga model cenderung overfitting. Pada eksperimen ini ditambahkan stochastic depth (`drop_path_rate` 0,2). Ukuran input diubah menjadi 512 x 384 agar sesuai rasio halaman, sehingga teks tidak terlalu terdistorsi dan resolusi vertikal lebih tinggi.

In [ ]:
outs["convnext_tiny_512"] = fit(configs["convnext_tiny_512"], tr_df, va_df, exp_name="convnext_tiny_512")
o = outs["convnext_tiny_512"]
print(f"Akurasi validation terbaik: {o['best_acc']:.4f} (epoch {o['best_epoch']}), waktu {o['minutes']:.1f} menit")
plot_history(o["history"], "ConvNeXt-Tiny (512 x 384)", FIG_DIR / "curve_convnext_tiny_512.png")

## 6. Ensemble
Probabilitas eksperimen 2 dan 3 dirata-ratakan. Kedua model melihat gambar pada ukuran yang berbeda, sehingga kesalahannya diharapkan tidak selalu sama.

In [ ]:
ENSEMBLE = ["convnext_tiny_384", "convnext_tiny_512"]
y_val = va_df["label"].values
ens_val = np.mean([outs[n]["val_probs"] for n in ENSEMBLE], axis=0)
ens_acc = (ens_val.argmax(1) == y_val).mean()

p2, p3 = (outs[n]["val_probs"].argmax(1) for n in ENSEMBLE)
print(f"Prediksi eksperimen 2 dan 3 berbeda pada {(p2 != p3).mean() * 100:.1f}% data validation")
print(f"Akurasi validation ensemble: {ens_acc:.4f}")

## 7. Perbandingan eksperimen

In [ ]:
rows = []
for name, out in outs.items():
    cfg, h = configs[name], out["history"]
    best = h.loc[h["epoch"] == out["best_epoch"]].iloc[0]
    hh, ww = to_hw(cfg["img_size"])
    rows.append({"eksperimen": name, "model": cfg["model_name"], "input": f"{hh}x{ww}", "epoch_dijalankan": len(h),
                 "epoch_terbaik": out["best_epoch"], "train_acc": best["train_acc"], "val_acc": out["best_acc"],
                 "val_loss": best["val_loss"], "selisih_train_val": best["train_acc"] - out["best_acc"], "menit": out["minutes"]})
rows.append({"eksperimen": "ensemble", "model": " + ".join(ENSEMBLE), "input": "-", "val_acc": ens_acc,
             "menit": sum(outs[n]["minutes"] for n in ENSEMBLE)})
experiments = pd.DataFrame(rows)
experiments.to_csv(METRIC_DIR / "experiments.csv", index=False)
display(experiments)

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, out in outs.items():
    ax.plot(out["history"]["epoch"], out["history"]["val_acc"], marker="o", label=name)
ax.axhline(ens_acc, color="black", ls="--", lw=1, label=f"ensemble ({ens_acc:.4f})")
ax.set(title="Akurasi validation per epoch", xlabel="epoch", ylabel="akurasi")
ax.grid(alpha=0.3)
ax.legend()
plt.savefig(FIG_DIR / "experiments_val_acc.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Prediksi validation
Probabilitas tiap model dan ensemble disimpan untuk analisis di `03_evaluation.ipynb`.

In [ ]:
val_probs = {name: out["val_probs"] for name, out in outs.items()}
val_probs["ensemble"] = ens_val
for name, probs in val_probs.items():
    pred_df = va_df[["id", "filename", "label"]].copy()
    pred_df["pred"] = probs.argmax(1)
    pred_df["confidence"] = probs.max(1)
    pred_df[[f"prob_{i}" for i in range(len(CLASS_NAMES))]] = probs
    pred_df.to_csv(METRIC_DIR / f"{name}_val_predictions.csv", index=False)
    print(f"{name}: akurasi validation {(pred_df['pred'] == pred_df['label']).mean():.4f}")

## 9. Submission

In [ ]:
test_probs = {}
for name, out in outs.items():
    model, cfg = load_checkpoint(out["ckpt_path"], device)
    test_probs[name] = predict_test(model, cfg, test_df)
    del model
    torch.cuda.empty_cache()
test_probs["ensemble"] = np.mean([test_probs[n] for n in ENSEMBLE], axis=0)

for name, probs in test_probs.items():
    np.save(METRIC_DIR / f"{name}_test_probs.npy", probs.astype(np.float32))
    sub = save_submission(probs, test_df, SUB_DIR / f"submission_{name}.csv")
    validate_submission(sub, test_df)
    print(f"submission_{name}.csv: {len(sub)} baris")

display(sub.head())
display(sub["label"].map(lambda i: CLASS_NAMES[i]).value_counts().to_frame("jumlah_prediksi").T)

## 10. Arsip hasil
Seluruh isi `results/` dan `checkpoint/` dikompres agar mudah diunduh dari tab Output Kaggle.

In [ ]:
import zipfile

archive = ROOT.parent / "hasil_training.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("results", "checkpoint"):
        for f in Path(folder).rglob("*"):
            if f.is_file():
                z.write(f)
print(archive, f"{archive.stat().st_size / 1e6:.1f} MB")